# インタラクティブ・トレーニング・ノートブック

このノートブックでは、チェックポイントから学習済みモデルを読み込み、
新規セッションのトレーニングを1ステップずつ実行できます。

## 機能
- チェックポイントからモデル読み込み
- セッションごとの段階的トレーニング
- トレーニング頻度の調整
- リアルタイムでの可視化

## 1. セットアップ

In [ ]:
import sys
import os

# プロジェクトルートをパスに追加
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import pandas as pd
from pathlib import Path

# プロジェクトのモジュールをインポート
from train import get_command_line_parser, initialize_trainer
from utils import set_seed, set_gpu

# Jupyter用の設定
%matplotlib inline
%load_ext autoreload
%autoreload 2

plt.style.use('seaborn-v0_8-darkgrid')

print(f"Project root: {project_root}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. トレーナーの初期化

In [ ]:
# コマンドライン引数を設定
parser = get_command_line_parser()
args = parser.parse_args([
    # '--dataset', 'CICIDS2017_improved',
    # '--encoder', 'cnn1d',
    # '--project', 'fact',
])

# シードを設定
set_seed(args.seed)

# トレーナーを初期化
print("Initializing trainer...")
trainer = initialize_trainer(args)

print(f"\nTrainer Configuration:")
print(f"  Dataset: {trainer.args.dataset}")
print(f"  Encoder: {trainer.args.encoder}")
print(f"  Project: {trainer.args.project}")
print(f"  Base classes: {trainer.args.base_class}")
print(f"  Total classes: {trainer.args.num_classes}")
print(f"  Way: {trainer.args.way}")
print(f"  Shot: {trainer.args.shot}")
print(f"  Total sessions: {trainer.args.sessions}")
print(f"  New sessions: {list(range(1, trainer.args.sessions))}")

## 3. チェックポイントから読み込み

In [ ]:
# 利用可能なチェックポイントを確認
checkpoint_dir = Path(trainer.args.save_path)

print(f"Checkpoint directory: {checkpoint_dir}")
print(f"\nAvailable checkpoints:")

if checkpoint_dir.exists():
    checkpoints = sorted(checkpoint_dir.glob('*.pth'))
    for i, cp in enumerate(checkpoints):
        print(f"  [{i}] {cp.name}")
else:
    print(f"  Warning: Directory not found. Please run base session training first.")

In [ ]:
# ベースセッションのチェックポイントを読み込み
base_checkpoint = checkpoint_dir / 'session0_max_acc.pth'

if base_checkpoint.exists():
    print(f"Loading checkpoint: {base_checkpoint}")
    checkpoint = torch.load(base_checkpoint)
    
    # モデルの状態を復元
    trainer.model.load_state_dict(checkpoint['params'])
    
    print(f"\nCheckpoint loaded successfully!")
    print(f"  Session: {checkpoint.get('session', 0)}")
    print(f"  Max accuracy: {checkpoint.get('max_acc', 'N/A')}")
    
    # ベースセッションのテスト
    print(f"\nTesting base session...")
    _, test_acc = trainer.test(trainer.model, 0, return_loss=False)
    print(f"Base session accuracy: {test_acc:.4f}")
else:
    print(f"Error: Checkpoint not found: {base_checkpoint}")
    print(f"Please run base session training first:")
    print(f"  dvc repro train_base")

## 4. トレーニングパラメータの設定

In [ ]:
# トレーニングパラメータをカスタマイズ
training_config = {
    'target_session': 1,          # トレーニングするセッション
    'max_epochs': 100,             # 最大エポック数
    'learning_rate': 0.1,          # 学習率
    'log_interval': 10,            # ログ出力間隔（エポック）
    'save_interval': 20,           # チェックポイント保存間隔
    'early_stopping_patience': 30, # Early stopping の patience
}

print("Training Configuration:")
for key, value in training_config.items():
    print(f"  {key}: {value}")

## 5. データローダーの準備

In [ ]:
# 指定されたセッションのデータローダーを取得
session = training_config['target_session']

print(f"Loading data for session {session}...")
trainset, trainloader, testloader = trainer.get_dataloader(session)

print(f"\nDataset info:")
print(f"  Training samples: {len(trainset)}")
print(f"  Training batches: {len(trainloader)}")
print(f"  Test batches: {len(testloader)}")

# サンプルデータを確認
if hasattr(trainset, 'targets'):
    unique_classes = np.unique(trainset.targets)
    print(f"  Classes in this session: {unique_classes}")

## 6. トレーニングループ（ステップバイステップ実行）

In [ ]:
# トレーニング履歴を記録
history = {
    'epoch': [],
    'train_loss': [],
    'train_acc': [],
    'test_acc': [],
}

# ベストモデルの追跡
best_acc = 0.0
patience_counter = 0

print(f"Starting training for session {session}...")
print(f"Press 'Interrupt kernel' to stop training.\n")

In [ ]:
# トレーニングを1エポックずつ実行
# このセルを繰り返し実行することで、段階的にトレーニングを進められます

if len(history['epoch']) < training_config['max_epochs']:
    current_epoch = len(history['epoch']) + 1
    
    # 1エポックのトレーニング
    trainer.model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    pbar = tqdm(trainloader, desc=f"Epoch {current_epoch}/{training_config['max_epochs']}")
    for batch_idx, batch in enumerate(pbar):
        if trainer.args.dataset == 'CICIDS2017_improved':
            data, target = batch
        else:
            data, target = batch[0], batch[1]
        
        data, target = data.cuda(), target.cuda()
        
        # フォワードパス
        trainer.optimizer.zero_grad()
        output = trainer.model(data)
        
        # ロス計算
        loss = trainer.criterion(output, target)
        
        # バックワード
        loss.backward()
        trainer.optimizer.step()
        
        # 統計
        train_loss += loss.item()
        _, predicted = output.max(1)
        train_total += target.size(0)
        train_correct += predicted.eq(target).sum().item()
        
        pbar.set_postfix({
            'loss': f"{train_loss/(batch_idx+1):.4f}",
            'acc': f"{100.*train_correct/train_total:.2f}%"
        })
    
    # テスト
    _, test_acc = trainer.test(trainer.model, session, return_loss=False)
    
    # 履歴に記録
    avg_train_loss = train_loss / len(trainloader)
    train_acc = 100. * train_correct / train_total
    
    history['epoch'].append(current_epoch)
    history['train_loss'].append(avg_train_loss)
    history['train_acc'].append(train_acc)
    history['test_acc'].append(test_acc)
    
    # ベストモデルの更新
    if test_acc > best_acc:
        best_acc = test_acc
        patience_counter = 0
        print(f"\n✓ New best accuracy: {best_acc:.4f}")
    else:
        patience_counter += 1
    
    # ログ出力
    if current_epoch % training_config['log_interval'] == 0:
        print(f"\nEpoch {current_epoch}:")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Train Acc: {train_acc:.2f}%")
        print(f"  Test Acc: {test_acc:.4f}")
        print(f"  Best Acc: {best_acc:.4f}")
        print(f"  Patience: {patience_counter}/{training_config['early_stopping_patience']}")
    
    # Early stopping
    if patience_counter >= training_config['early_stopping_patience']:
        print(f"\n⚠ Early stopping triggered. No improvement for {patience_counter} epochs.")
else:
    print(f"Training completed! Maximum epochs reached.")

## 7. トレーニング履歴の可視化

In [ ]:
if len(history['epoch']) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss
    axes[0].plot(history['epoch'], history['train_loss'], 'b-', label='Train Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    # Accuracy
    axes[1].plot(history['epoch'], history['train_acc'], 'b-', label='Train Acc')
    axes[1].plot(history['epoch'], [acc*100 for acc in history['test_acc']], 'r-', label='Test Acc')
    axes[1].axhline(y=best_acc*100, color='g', linestyle='--', label=f'Best: {best_acc*100:.2f}%')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Training and Test Accuracy')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nTraining Statistics:")
    print(f"  Epochs completed: {len(history['epoch'])}")
    print(f"  Best test accuracy: {best_acc:.4f}")
    print(f"  Latest test accuracy: {history['test_acc'][-1]:.4f}")
else:
    print("No training history yet. Run the training cell above.")

## 8. チェックポイントの保存

In [ ]:
# 現在のモデルをチェックポイントとして保存
save_path = checkpoint_dir / f'session{session}_notebook_{len(history["epoch"])}ep.pth'

checkpoint = {
    'session': session,
    'epoch': len(history['epoch']),
    'params': trainer.model.state_dict(),
    'max_acc': best_acc,
    'history': history,
}

torch.save(checkpoint, save_path)
print(f"Checkpoint saved: {save_path}")
print(f"  Session: {session}")
print(f"  Epochs: {len(history['epoch'])}")
print(f"  Best accuracy: {best_acc:.4f}")

## 9. 全セッションのテスト

In [ ]:
# 全セッションでテスト
print("Testing on all sessions...\n")

session_results = {}
for test_session in range(session + 1):
    _, acc = trainer.test(trainer.model, test_session, return_loss=False)
    session_results[test_session] = acc
    print(f"Session {test_session} accuracy: {acc:.4f}")

# 結果を可視化
plt.figure(figsize=(10, 5))
sessions = list(session_results.keys())
accuracies = [session_results[s]*100 for s in sessions]

plt.bar(sessions, accuracies, alpha=0.7, color='steelblue')
plt.xlabel('Session')
plt.ylabel('Accuracy (%)')
plt.title('Test Accuracy on All Sessions')
plt.xticks(sessions)
plt.grid(True, axis='y', alpha=0.3)

# 各バーの上に値を表示
for s, acc in zip(sessions, accuracies):
    plt.text(s, acc + 1, f'{acc:.2f}%', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print(f"\nAverage accuracy: {np.mean(accuracies):.2f}%")

## 10. バッチトレーニング（自動実行）

複数エポックを自動的に実行する場合は、このセルを使用します。

In [ ]:
# バッチトレーニング設定
batch_config = {
    'num_epochs': 50,  # 実行するエポック数
}

print(f"Running {batch_config['num_epochs']} epochs automatically...\n")

for _ in range(batch_config['num_epochs']):
    if len(history['epoch']) >= training_config['max_epochs']:
        print("Maximum epochs reached!")
        break
    
    if patience_counter >= training_config['early_stopping_patience']:
        print("Early stopping triggered!")
        break
    
    current_epoch = len(history['epoch']) + 1
    
    # 1エポックのトレーニング（上記と同じロジック）
    trainer.model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for batch in trainloader:
        if trainer.args.dataset == 'CICIDS2017_improved':
            data, target = batch
        else:
            data, target = batch[0], batch[1]
        
        data, target = data.cuda(), target.cuda()
        
        trainer.optimizer.zero_grad()
        output = trainer.model(data)
        loss = trainer.criterion(output, target)
        loss.backward()
        trainer.optimizer.step()
        
        train_loss += loss.item()
        _, predicted = output.max(1)
        train_total += target.size(0)
        train_correct += predicted.eq(target).sum().item()
    
    # テスト
    _, test_acc = trainer.test(trainer.model, session, return_loss=False)
    
    # 履歴に記録
    avg_train_loss = train_loss / len(trainloader)
    train_acc = 100. * train_correct / train_total
    
    history['epoch'].append(current_epoch)
    history['train_loss'].append(avg_train_loss)
    history['train_acc'].append(train_acc)
    history['test_acc'].append(test_acc)
    
    # ベストモデルの更新
    if test_acc > best_acc:
        best_acc = test_acc
        patience_counter = 0
        print(f"Epoch {current_epoch}: Test Acc {test_acc:.4f} ✓ (new best)")
    else:
        patience_counter += 1
        if current_epoch % 10 == 0:
            print(f"Epoch {current_epoch}: Test Acc {test_acc:.4f} (best: {best_acc:.4f})")

print(f"\nBatch training completed!")
print(f"  Total epochs: {len(history['epoch'])}")
print(f"  Best accuracy: {best_acc:.4f}")

## 11. データ分析

In [ ]:
# トレーニング履歴をDataFrameに変換
if len(history['epoch']) > 0:
    df_history = pd.DataFrame(history)
    df_history['test_acc_pct'] = df_history['test_acc'] * 100
    
    print("Training History (last 10 epochs):")
    print(df_history.tail(10).to_string(index=False))
    
    print(f"\nSummary Statistics:")
    print(df_history[['train_loss', 'train_acc', 'test_acc_pct']].describe())